In [2]:
#  Ask the operating system what video card is plugged in
!nvidia-smi

# Check if i am using a100 gpu
import torch

if torch.cuda.is_available():
    print(f"Success! Connected to: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU only")

zsh:1: command not found: nvidia-smi


In [3]:
from sklearn.model_selection import train_test_split
from scipy.spatial import KDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from pathlib import Path
import pandas as pd
import numpy as np
import os
import glob

In [ ]:
# Configuration
bus_file    =      '../../data/clean_data/bus/clean_bus_df.csv'
subway_file =      '../../data/clean_data/sub/cleaned_subway_entrances.csv'
output_file =      '../../data/joined/augmented_properties_with_distances.csv'
chunk_size  =      100000  # Adjust for RAM (smaller = slower but safer)
total_rows  =      0

# Creates joined folder if it doesn't exist
path_obj = Path(output_file)
path_obj.parent.mkdir(parents=True, exist_ok=True)

# Load data
buses = pd.read_csv(bus_file)
subways = pd.read_csv(subway_file)

# Prep coords
bus_coords = np.column_stack((buses['Latitude'], buses['Longitude']))
subway_coords = np.column_stack((subways['Entrance Latitude'], subways['Entrance Longitude']))

# Build KD Trees (very fast look up)
bus_tree = KDTree(bus_coords)
subway_tree = KDTree(subway_coords)

# Clear output if exists
if os.path.exists(output_file):
    os.remove(output_file)

In [5]:
# Haversine formula for distance in meters

def haversine(lon1: np.ndarray, lat1: np.ndarray, lon2: np.ndarray, lat2: np.ndarray) -> float:
    R = 6371000  # Radius of Earth in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

In [6]:
print(f"Processing cleaned_property_part_*.csv files in chunks of {chunk_size}...")

dtypes = {
    'BBL': 'object',
    'BORO': 'Int64',
    'TAXCLASS': 'Int64',
    'FULLVAL': 'Int64',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'NTA': 'category',
    'Borough': 'category'
}


part_files = sorted(glob.glob('../../data/clean_data/val/cleaned_property_part_*.csv'))
print(f"Found {len(part_files)} files to process.")

for part_file in part_files:
    part_num = part_file.split('_')[-1].replace('.csv', '')  # Extract number for logging
    print(f"Starting part {part_num}: {part_file}")
    
    # Process each part file in chunks
    for chunk_idx, chunk in enumerate(pd.read_csv(part_file, chunksize=chunk_size, low_memory=False, dtype=dtypes)):
        print(f"  Chunk {chunk_idx + 1} in part {part_num}: {len(chunk)} rows")
        
        prop_coords = np.column_stack((chunk['Latitude'], chunk['Longitude']))
        
        # Nearest bus
        _, bus_idx = bus_tree.query(prop_coords, k=1)
        chunk['dist_to_bus_m'] = [haversine(chunk.iloc[i]['Latitude'], chunk.iloc[i]['Longitude'],
                                            buses.iloc[idx]['Latitude'], buses.iloc[idx]['Longitude'])
                                  for i, idx in enumerate(bus_idx.flatten())]
        chunk['nearest_bus_nta'] = [buses.iloc[idx]['NTAName'] for idx in bus_idx.flatten()]  # Optional extra
        
        # Nearest subway
        _, sub_idx = subway_tree.query(prop_coords, k=1)
        chunk['dist_to_subway_m'] = [haversine(chunk.iloc[i]['Latitude'], chunk.iloc[i]['Longitude'],
                                               subways.iloc[idx]['Entrance Latitude'], subways.iloc[idx]['Entrance Longitude'])
                                     for i, idx in enumerate(sub_idx.flatten())]
        chunk['nearest_subway_routes'] = [subways.iloc[idx]['Routes'] for idx in sub_idx.flatten()]  # Optional extra
        
        # Select final columns (keep originals + new ones)
        final_cols = ['BBL', 'BORO', 'TAXCLASS', 'FULLVAL', 'Latitude', 'Longitude', 'NTA', 'Borough',
                      'dist_to_bus_m', 'dist_to_subway_m', 'nearest_bus_nta', 'nearest_subway_routes']
        chunk_final = chunk[final_cols]
        
        # Append to output
        mode = 'a' if total_rows > 0 else 'w'
        header = total_rows == 0
        chunk_final.to_csv(output_file, mode=mode, header=header, index=False)
        total_rows += len(chunk_final)

print(f"Done! Output: {output_file} with {total_rows} rows")

Processing cleaned_property_part_*.csv files in chunks of 100000...
Found 8 files to process.
Starting part 1: ../../data/clean_data/val/cleaned_property_part_1.csv
  Chunk 1 in part 1: 100000 rows
  Chunk 2 in part 1: 100000 rows
  Chunk 3 in part 1: 100000 rows
  Chunk 4 in part 1: 100000 rows
  Chunk 5 in part 1: 100000 rows
  Chunk 6 in part 1: 100000 rows
  Chunk 7 in part 1: 100000 rows
  Chunk 8 in part 1: 100000 rows
  Chunk 9 in part 1: 100000 rows
  Chunk 10 in part 1: 100000 rows
Starting part 2: ../../data/clean_data/val/cleaned_property_part_2.csv
  Chunk 1 in part 2: 100000 rows
  Chunk 2 in part 2: 100000 rows
  Chunk 3 in part 2: 100000 rows
  Chunk 4 in part 2: 100000 rows
  Chunk 5 in part 2: 100000 rows
  Chunk 6 in part 2: 100000 rows
  Chunk 7 in part 2: 100000 rows
  Chunk 8 in part 2: 100000 rows
  Chunk 9 in part 2: 100000 rows
  Chunk 10 in part 2: 100000 rows
Starting part 3: ../../data/clean_data/val/cleaned_property_part_3.csv
  Chunk 1 in part 3: 100000 row

In [ ]:
# Evaluation  --- ONLY ON THE FIRST 100K

df = pd.read_csv('../../data/joined/augmented_properties_with_distances.csv', nrows=100000, low_memory=False)
df['log_fullval'] = np.log(df['FULLVAL'] + 1)

# New features
df['log_dist_bus'] = np.log(df['dist_to_bus_m'] + 1)  # + 1 avoids log(0)
df['log_dist_subway'] = np.log(df['dist_to_subway_m'] + 1)
df['num_subway_routes'] = df['nearest_subway_routes'].str.split(',').str.len().fillna(0)  # Count routes

# Select expanded X - drop any NaNs after deriving
X = df[['log_dist_bus', 'log_dist_subway', 'TAXCLASS', 'num_subway_routes', 'NTA']].copy()
y = df['log_fullval']

# Handle categories: One hot NTA (top 20 for simplicity --- full would use all)
top_ntas = X['NTA'].value_counts().head(20).index
X['NTA'] = X['NTA'].apply(lambda x: x if x in top_ntas else 'Other')

# Drop NaNs
mask = ~(X.isna().any(axis=1)) & y.notna()
X_clean = X[mask]
y_clean = y[mask]
print(f"Shape after dropping NaNs: {X_clean.shape} (dropped {len(X) - len(X_clean)} rows)")

# Pipeline for preprocessing 
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(drop='first', sparse_output=False), ['NTA'])], remainder='passthrough')

# Split and fit pipeline with Random Forest
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)
model = Pipeline([('preproc', preprocessor),('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
model.fit(X_train, y_train)

# Predict and score
y_pred = model.predict(X_test)
print(f"R-squared score on test set: {r2_score(y_test, y_pred):.4f}")

# Feature importances - see if we can extract any insights
importances = model.named_steps['rf'].feature_importances_
feature_names = (['NTA_' + name for name in top_ntas[1:]] +  # Skip first for drop
                 ['log_dist_bus', 'log_dist_subway', 'TAXCLASS', 'num_subway_routes'])
print("Top feature importances:", sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:5])

Shape after dropping NaNs: (85731, 5) (dropped 14269 rows)
R-squared score on test set: 0.6890
Top feature importances: [('TAXCLASS', np.float64(0.2912698357135899)), ('log_dist_subway', np.float64(0.2627327389655626)), ('num_subway_routes', np.float64(0.19064463237211945)), ('NTA_Turtle Bay-East Midtown', np.float64(0.018704652189868116)), ('NTA_Windsor Terrace', np.float64(0.017867098923821646))]
